# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to inspect and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) containing mass spectrometry results for extracellular vesicles isolated from stimulated human mast cells.

### Dataset Source
The dataset is defined by a Croissant JSON-LD file accessible via:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset (metadata and access to records)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# Explore the structure of the dataset
# Get all record sets and their fields (@ids)

record_sets = []
try:
    # Usually, record sets are available via metadata.record_sets or metadata.recordSet
    # If both exist, prefer record_sets (plural). Fallback to recordSet (singular/plural depending on schema version)
    rs_attr = None
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        rs_attr = 'record_sets'
    elif hasattr(metadata, 'recordSet') and metadata.recordSet:
        rs_attr = 'recordSet'
    if not rs_attr:
        raise AttributeError("No record sets found in metadata.")
    record_sets = getattr(metadata, rs_attr)
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- Record set @id: {rs['@id']}")
        fields = rs.get('fields', rs.get('field', []))
        print("  Fields in this record set:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)} (name: {field.get('name')})")
            else:
                print(f"    - {field}")
        print()
except Exception as e:
    # Some Croissant schemas store record sets in metadata.to_json()['recordSet'] as a list of dicts
    # For the FAIR^2 dataset, we know from the provided package it is likely empty, so fetch by parsing the JSON
    j = dataset.metadata.to_json()
    record_sets_json = j.get('recordSet', [])
    print(f"Fallback: Found {len(record_sets_json)} record set(s) in JSON metadata.")
    for rs in record_sets_json:
        print(f"- Record set @id: {rs['@id']}")
        fields = rs.get('field', [])
        print("  Fields in this record set:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', field)} (name: {field.get('name')})")
            else:
                print(f"    - {field}")
        print()
    if not record_sets_json:
        # Try to find record set info by checking distribution DataDownloads
        print("\nNo explicit record sets found in the schema.\nTrying to infer record sets from available distributions...")
        distribution = j.get('distribution', [])
        for file in distribution:
            if isinstance(file, dict):
                print(f"- Data file @id: {file['@id']} (potential record set)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the record set `@id` value from above. For this dataset, the main table is in the following record set:

`cr:EVSummary`

In [ ]:
# Extract data from the main record set
# In FAIR², the main record set for protein summaries is 'cr:EVSummary'
main_record_set_id = 'cr:EVSummary'

# You may list multiple record sets if desired, e.g. ['cr:EVSummary', 'cr:AnotherSet']
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set])} rows from record set '{record_set}'.")
        print(f"Columns for '{record_set}':\n{dataframes[record_set].columns.tolist()}")
    except Exception as err:
        print(f"Could not load record set {record_set}: {err}")

display_cols = dataframes[main_record_set_id].columns.tolist() if main_record_set_id in dataframes else []
# Display first few rows
if main_record_set_id in dataframes:
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, or grouping.

Let's analyze the `MW (kDa)` field (molecular weight, in kilodaltons), using its column name `cr:MWkDa`, and see how proteins group by the `cr:Gene` field, if present.

In [ ]:
# Choose a numeric field for analysis. In this dataset, molecular weight is common.
record_set_id = main_record_set_id

# Identify the MW (kDa) field using the @id convention -- assume it's 'cr:MWkDa'
numeric_field_id = 'cr:MWkDa'  # The @id of molecular weight column

df = dataframes[record_set_id]

# Check for missing entries or type issues
df = df.copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 50  # Only review large proteins for demo
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered proteins with molecular weight > {threshold} kDa:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by gene symbol if available
group_field_id = 'cr:Gene'  # Field representing gene symbol
if group_field_id in df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean MW (kDa) by gene symbol (top rows):")
    print(grouped.head())

## 5. Visualization
Visualize molecular weight distribution and the top gene groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of molecular weights
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title('Distribution of Protein MW (kDa)')
plt.xlabel('MW (kDa)')
plt.ylabel('Count')
plt.show()

# Top 10 genes by average MW
if group_field_id in df.columns:
    avg_mw_by_gene = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=avg_mw_by_gene.index, y=avg_mw_by_gene.values)
    plt.xticks(rotation=45)
    plt.title('Top 10 Genes by Mean MW (kDa)')
    plt.ylabel('Mean MW (kDa)')
    plt.xlabel('Gene')
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² protein mass spectrometry dataset using Croissant's structure via the `mlcroissant` library.
We demonstrated how to find record set and field `@id`s, load tables, filter and normalize numeric features, group by categories, and visualize protein molecular weight distributions and gene averages. 

For a full analysis, you may extend this notebook to cover additional fields, filters, or visualizations, and consult the dataset's schema at the [Croissant source URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

#### References
- [mlcroissant library](https://github.com/mlcommons/croissant)
- [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- [Croissant specification](https://mlcommons.org/croissant/spec/)
